# 06 — Private Set Submission

Run 2-phase adaptive inference on the private test set and produce the submission CSV.

**Pipeline order:**
1. `02_inference.ipynb` — full public inference (`N_QUESTIONS=None`)
2. `03_qlora_finetune.ipynb` — QLoRA SFT → merge adapter
3. `04_grpo_train.ipynb` — GRPO RL → merge adapter
4. **This notebook** → `artifacts/submissions/submission_*.csv`

The notebook auto-selects the best merged model: GRPO > QLoRA > base.
No scoring section — the private set has no ground-truth labels.


In [1]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# No-op when running locally.  On Colab: installs packages, mounts Drive.
import sys

try:
    import google.colab  # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

if _IS_COLAB:
    print("Colab detected — installing packages (takes ~3-5 min)...")
    import subprocess
    pkgs = [
        "vllm>=0.6.0", "transformers>=4.45.0", "bitsandbytes>=0.43.0",
        "peft>=0.12.0", "trl>=0.12.0", "datasets", "accelerate>=0.34.0",
        "sympy>=1.12", "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("Packages installed; mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted. Upload public.jsonl / private.jsonl to")
    print("  MyDrive/math-qa-llm/data/raw/  before running further cells.")
else:
    print("Running locally — skipping Colab setup.")


Running locally — skipping Colab setup.


## 1. Config & Imports

Paths, token budgets, and batching parameters for the private run.

- **Model auto-selection:** prefers `grpo_v1_merged` → `qlora_v1_merged` → base Qwen3-4B. Whichever merged checkpoint exists is used automatically.
- **vLLM is disabled** — DSMLP CUDA 12.8 is incompatible with vLLM 0.21.0 (compiled for CUDA 13). HuggingFace Transformers is used instead.
- **`CHUNK_SIZE=6`** — number of prompts batched per `model.generate()` call; tuned for A30 24 GB BF16 KV-cache budget.

In [2]:
import json, os, re, sys
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))

GPU_ID = "0"

_GRPO_MERGED  = REPO_ROOT / "artifacts" / "models" / "grpo_v1_merged"
_QLORA_MERGED = REPO_ROOT / "artifacts" / "models" / "qlora_v1_merged"
_BASE_MODEL   = "Qwen/Qwen3-4B-Thinking-2507"

if (_GRPO_MERGED / "config.json").is_file():
    MODEL_ID, MODEL_LABEL = str(_GRPO_MERGED), "grpo_v1_merged"
elif (_QLORA_MERGED / "config.json").is_file():
    MODEL_ID, MODEL_LABEL = str(_QLORA_MERGED), "qlora_v1_merged"
else:
    MODEL_ID, MODEL_LABEL = _BASE_MODEL, "base_model"


PRIVATE_PATH = REPO_ROOT / "data" / "raw" / "private.jsonl"

RUN_NAME        = f"private_{MODEL_LABEL}"
OUTPUT_PATH     = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
CHECKPOINT_PATH = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"

# ── Phase 1 — fast first pass ────────────────────────────────────────────────
PHASE1_THINKING_BUDGET    = 1024
PHASE1_MAX_TOKENS         = 4096
PHASE1_N_SAMPLES          = 1

# ── Phase 2 — retry on questions flagged uncertain after Phase 1 ─────────────
PHASE2_THINKING_BUDGET    = 4096   # keep full budget for uncertain questions
PHASE2_MAX_TOKENS         = 5120
PHASE2_N_SAMPLES          = 3
PHASE2_TEMPERATURE        = 0.65
PHASE2_REPETITION_PENALTY = 1.05

# ── Phase 2 cross-question batching (NEW — big speedup) ──────────────────────
# Like Phase 1 batches CHUNK_SIZE different questions per generate() call, Phase
# 2 now batches PHASE2_BATCH_QUESTIONS different questions × PHASE2_N_SAMPLES
# samples per call. Sharing GPU compute across multiple questions amortizes the
# per-question overhead.
# Optimal pick: PHASE2_BATCH_QUESTIONS × PHASE2_N_SAMPLES == CHUNK_SIZE
#   With CHUNK_SIZE=12 and N=3 → K=4   (12 sequences per call)
# Larger K reduces per-question overhead but the slowest-finishing sequence in
# each batch dictates total batch time.
PHASE2_BATCH_QUESTIONS = 3

# ── Phase 3 — DISABLED for time-safe submission ──────────────────────────────
USE_PHASE3                = False
PHASE3_THINKING_BUDGET    = 6144
PHASE3_MAX_TOKENS         = 8192
PHASE3_N_SAMPLES          = 8
PHASE3_TEMPERATURE        = 0.8
PHASE3_REPETITION_PENALTY = 1.10

MAX_TOKENS = PHASE3_MAX_TOKENS if USE_PHASE3 else PHASE2_MAX_TOKENS

# ── Transformers batching ─────────────────────────────────────────────────────
# CHUNK_SIZE is the maximum number of prompts processed per model.generate() call.
# Phase 1 batches CHUNK_SIZE different questions per call.
# Phase 2 batches PHASE2_BATCH_QUESTIONS × PHASE2_N_SAMPLES sequences per call
# (designed to fit exactly inside CHUNK_SIZE).
# 12 works on L40S 48 GB with NF4 model.
CHUNK_SIZE = 9

# ── EBM verifier config ───────────────────────────────────────────────────────
USE_EBM  = True
EBM_PATH = REPO_ROOT / "artifacts" / "models" / "ebm_verifier_v1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    DRIVE_BASE    = Path('/content/drive/MyDrive/math-qa-llm')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    _GRPO_MERGED  = DRIVE_BASE / "artifacts" / "models" / "grpo_v1_merged"
    _QLORA_MERGED = DRIVE_BASE / "artifacts" / "models" / "qlora_v1_merged"
    if (_GRPO_MERGED / "config.json").is_file():
        MODEL_ID, MODEL_LABEL = str(_GRPO_MERGED), "grpo_v1_merged"
    elif (_QLORA_MERGED / "config.json").is_file():
        MODEL_ID, MODEL_LABEL = str(_QLORA_MERGED), "qlora_v1_merged"
    PRIVATE_PATH    = DRIVE_BASE / "data" / "raw" / "private.jsonl"
    OUTPUT_PATH     = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
    CHECKPOINT_PATH = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"
    EBM_PATH        = DRIVE_BASE / "artifacts" / "models" / "ebm_verifier_v1"
    (DRIVE_BASE / "artifacts" / "logs" / "runs").mkdir(parents=True, exist_ok=True)

from transformers import AutoTokenizer
from tqdm.auto import tqdm

# vLLM disabled — DSMLP CUDA 12.8 vs vLLM 0.21.0 ELF symbol mismatch
VLLM_AVAILABLE = False
LLM            = None
SamplingParams = None
USE_VLLM       = False
print("vLLM disabled — using HuggingFace Transformers (DSMLP CUDA 12.8 incompatibility).")

assert PRIVATE_PATH.is_file(), f"private.jsonl not found: {PRIVATE_PATH}"
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"MODEL: {MODEL_LABEL}")
print(f"PRIVATE_PATH: {PRIVATE_PATH}")
print(f"CHECKPOINT: {CHECKPOINT_PATH}")
print(f"PHASE1: think={PHASE1_THINKING_BUDGET} max={PHASE1_MAX_TOKENS}")
print(f"PHASE2: think={PHASE2_THINKING_BUDGET} max={PHASE2_MAX_TOKENS} "
      f"N={PHASE2_N_SAMPLES} batch_questions={PHASE2_BATCH_QUESTIONS} "
      f"(batched call = {PHASE2_BATCH_QUESTIONS * PHASE2_N_SAMPLES} sequences)")
if USE_PHASE3:
    print(f"PHASE3: think={PHASE3_THINKING_BUDGET} max={PHASE3_MAX_TOKENS} "
          f"N={PHASE3_N_SAMPLES} temp={PHASE3_TEMPERATURE}  (ENABLED)")
else:
    print("PHASE3: DISABLED — Phase 2 responses go to CSV for any still-uncertain questions")
print(f"CHUNK_SIZE: {CHUNK_SIZE}")
print(f"USE_EBM: {USE_EBM}")
print(f"vLLM available: {VLLM_AVAILABLE}")

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vLLM disabled — using HuggingFace Transformers (DSMLP CUDA 12.8 incompatibility).
REPO_ROOT: /home/ssobirov/math-qa-llm
MODEL: qlora_v1_merged
PRIVATE_PATH: /home/ssobirov/math-qa-llm/data/raw/private.jsonl
CHECKPOINT: /home/ssobirov/math-qa-llm/artifacts/logs/runs/private_qlora_v1_merged_checkpoint.jsonl
PHASE1: think=1024 max=4096
PHASE2: think=4096 max=5120 N=3 batch_questions=3 (batched call = 9 sequences)
PHASE3: DISABLED — Phase 2 responses go to CSV for any still-uncertain questions
CHUNK_SIZE: 9
USE_EBM: True
vLLM available: False


## 2. Load Private Dataset

Loads all rows from `data/raw/private.jsonl`. The private set has no `answer` field — this notebook skips scoring entirely and goes straight to generation.

In [3]:
with open(PRIVATE_PATH, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

data_run = data  # always all private questions

n_mcq = sum(bool(d.get("options")) for d in data)
print(f"Loaded {len(data)} private questions ({n_mcq} MCQ, {len(data)-n_mcq} free-form)")


Loaded 943 private questions (300 MCQ, 643 free-form)


## 3. Prompt Construction

Two system prompts matching the format used in training:
- **Free-form** (`SYSTEM_PROMPT_MATH`): UNDERSTAND → PLAN → SOLVE → VERIFY → ANSWER structure with `\boxed{}` output
- **MCQ** (`SYSTEM_PROMPT_MCQ`): same structure plus ELIMINATE step; answer must be a single letter in `\boxed{X}`

Multi-answer free-form questions (`[ANS]` count > 1) get a hint injected into the user message.

In [4]:
from typing import Optional

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You'd better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You'd better be sure of your answer."
)


def build_prompt(question: str, options: Optional[list]) -> tuple:
    if options:
        labels   = [chr(65 + i) for i in range(len(options))]
        opts_txt = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_txt}"
    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint
    return SYSTEM_PROMPT_MATH, question


## 4. Generation Helpers

Key functions used by the generation loop:

| Function | Purpose |
|----------|---------|
| `extract_boxed()` | Extracts the last `\boxed{...}` from a response (handles nested braces) |
| `strip_thinking()` | Removes `<think>…</think>` block to get the answer-only section |
| `is_uncertain()` | Flags a response as uncertain if: truncated (`finish_reason="length"`), no `\boxed{}`, or answer section < 30 chars |
| `choose_best_sample()` | Majority vote over N samples; tie-break by longest response |
| `make_sampling_params()` | Returns a `model.generate()` kwargs dict (replaces vLLM `SamplingParams`) |
| `generate_batch()` | Runs inference in chunks with left-padded batching; returns `{"text", "finish_reason"}` per sequence |
| `build_chat_prompt()` | Applies the Qwen3 chat template with `enable_thinking=True` and optional `thinking_budget` |
| `load_checkpoint()` / `write_checkpoint()` | JSONL checkpoint for crash-safe resumable runs |

In [5]:
import torch
import torch.nn as nn
from collections import Counter
from math import ceil


def extract_boxed(text: str) -> str:
    matches, needle, i = [], r"\boxed{", 0
    while i < len(text):
        idx = text.find(needle, i)
        if idx == -1:
            break
        j, depth, start = idx + len(needle), 1, idx + len(needle)
        while j < len(text) and depth:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    matches.append(text[start:j])
                    break
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


def strip_thinking(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def is_uncertain(response: str, finish_reason: str = "") -> bool:
    if "length" in str(finish_reason).lower():
        return True
    if not extract_boxed(response):
        return True
    if len(strip_thinking(response)) < 30:
        return True
    return False


def choose_best_sample(samples: list, finish_reasons: list) -> dict:
    extracted = [extract_boxed(s) for s in samples]
    nonempty  = [e for e in extracted if e]
    if nonempty:
        counts    = Counter(nonempty)
        top_count = counts.most_common(1)[0][1]
        tied      = {a for a, c in counts.items() if c == top_count}
        candidates = [i for i, a in enumerate(extracted) if a in tied]
        best_idx  = max(candidates, key=lambda i: len(samples[i]))
        best_ans  = extracted[best_idx]
    else:
        top_count, best_idx, best_ans = 0, 0, ""
    uncertain = (is_uncertain(samples[best_idx], finish_reasons[best_idx])
                 or top_count < ceil(len(samples) / 2))
    return {"response": samples[best_idx], "answer": best_ans,
            "finish_reason": finish_reasons[best_idx], "consensus_count": top_count,
            "n_samples": len(samples), "uncertain": uncertain}


def make_sampling_params(max_tokens: int, temperature: float = 0.6,
                         repetition_penalty: float = 1.0) -> dict:
    """Return model.generate() kwargs dict — replaces vLLM SamplingParams."""
    return dict(
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        top_k=20,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )


SDPA_RUNTIME_ERROR_HINTS = (
    "scaled_dot_product_attention",
    "sdpa_attention_forward",
    "mha_graph.execute",
    "Expected mha_graph.execute",
)


def looks_like_sdpa_runtime_error(exc: RuntimeError) -> bool:
    message = str(exc)
    return any(hint in message for hint in SDPA_RUNTIME_ERROR_HINTS)


def run_generation_chunk(chunk: list, gen_kwargs: dict) -> list:
    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=16384,
    ).to(llm.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out_ids = llm.generate(**inputs, **gen_kwargs)

    new_ids = out_ids[:, prompt_len:]
    outputs = []
    for seq in new_ids:
        last_tok = seq[-1].item()
        finish = "length" if last_tok != tokenizer.eos_token_id else "stop"
        text = tokenizer.decode(seq, skip_special_tokens=True).strip()
        outputs.append({"text": text, "finish_reason": finish})

    del inputs, out_ids, new_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return outputs


def generate_batch(prompts: list, gen_kwargs: dict, chunk_size: int = CHUNK_SIZE) -> list:
    """Batch-generate with a DSMLP-safe fallback for attention backend failures."""
    results = []
    for i in range(0, len(prompts), chunk_size):
        chunk = prompts[i : i + chunk_size]
        try:
            results.extend(run_generation_chunk(chunk, gen_kwargs))
        except RuntimeError as exc:
            if looks_like_sdpa_runtime_error(exc) and len(chunk) > 1:
                print("Retrying the batch one prompt at a time after an attention backend failure.")
                for prompt in chunk:
                    results.extend(run_generation_chunk([prompt], gen_kwargs))
                continue
            if looks_like_sdpa_runtime_error(exc):
                raise RuntimeError(
                    "Generation failed inside the attention backend. "
                    "Reload the model with ATTN_IMPLEMENTATION='eager' on DSMLP and retry."
                ) from exc
            raise
    return results


def build_chat_prompt(item: dict, thinking_budget=None,
                      prefix: str = "", suffix: str = "") -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    if prefix:
        user = prefix + user
    if suffix:
        user = user + suffix
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=True)
    if thinking_budget is not None:
        kwargs["thinking_budget"] = thinking_budget
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        if thinking_budget is None:
            raise
        hint = (f"Use at most about {thinking_budget} thinking tokens. "
                "Be concise but do not skip necessary arithmetic.\n\n")
        messages[1]["content"] = hint + messages[1]["content"]
        kwargs.pop("thinking_budget", None)
        return tokenizer.apply_chat_template(messages, **kwargs)


def load_checkpoint(path) -> dict:
    if not path.exists():
        return {}
    records = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records[str(rec["id"])] = rec
    print(f"Loaded {len(records)} checkpoint records from {path.name}")
    return records


def write_checkpoint(path, records: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in sorted(records.values(), key=lambda r: int(r["id"])):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


# ── EBM verifier helpers ──────────────────────────────────────────────────────
# CRITICAL: VerifierHead.score() calls base_model.model (the inner Qwen3Model)
# NOT base_model (Qwen3ForCausalLM). This skips the lm_head 150K-vocab projection
# which would allocate ~10 GB of logits per scoring call and OOM the A30 24 GB.
# The EBM only needs hidden states — logits are irrelevant.
class VerifierHead(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.head = nn.Linear(hidden_size, 1, bias=True)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def score(self, base_model, input_ids, attention_mask) -> torch.Tensor:
        """(batch,) scalar scores. No gradients on base_model."""
        # Use inner backbone — bypasses lm_head, returns last_hidden_state directly.
        backbone = base_model.model if hasattr(base_model, "model") else base_model
        with torch.no_grad():
            hidden = backbone(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            ).last_hidden_state                            # (batch, seq_len, hidden)
        seq_lengths       = attention_mask.sum(dim=1) - 1   # (batch,)
        last_token_hidden = hidden[
            torch.arange(hidden.size(0), device=hidden.device),
            seq_lengths,
        ]                                                    # (batch, hidden)
        return self.head(last_token_hidden).squeeze(-1)      # (batch,)


# ── EBM scoring batching knobs (prevent OOM on 24 GB GPUs) ────────────────────
# Why this matters: each EBM forward pass needs to materialize attention activations
# proportional to (batch × seq_len²). At batch=4, seq=4000 the SDPA mem-efficient
# backend can still spike to 16+ GB which OOMs alongside the 8 GB model.
# Process candidates 2 at a time keeps peak EBM scoring memory under ~4 GB.
EBM_SCORE_BATCH   = 2      # candidates per scoring forward pass
EBM_SCORE_MAX_LEN = 3072   # truncate scoring input; \boxed{} is at the END so we
                            # use right-truncation (drops the start of <think>, not the answer)


def score_candidates_ebm(item: dict, candidates: list, verifier_head) -> list:
    """Score all candidates using VerifierHead + frozen llm in mini-batches.

    Mini-batched to keep peak attention memory bounded on 24 GB GPUs.
    Falls back to one-at-a-time scoring if even a mini-batch OOMs (very rare).
    """
    _system = "You are evaluating whether this mathematical solution is correct and complete."
    _user   = item["question"]
    if item.get("options"):
        _labels = [chr(65 + i) for i in range(len(item["options"]))]
        _user  += "\n\nOptions:\n" + "\n".join(
            f"{l}. {o.strip()}" for l, o in zip(_labels, item["options"])
        )

    # Build the full text for each candidate
    texts = []
    for cand in candidates:
        msgs = [
            {"role": "system",    "content": _system},
            {"role": "user",      "content": _user},
            {"role": "assistant", "content": cand},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))

    # Right-padding for last-token pooling; restored in the finally block
    _saved_side            = tokenizer.padding_side
    tokenizer.padding_side = "right"

    all_scores = []
    try:
        for start in range(0, len(texts), EBM_SCORE_BATCH):
            mini_batch = texts[start : start + EBM_SCORE_BATCH]
            enc = tokenizer(
                mini_batch, padding=True, truncation=True,
                max_length=EBM_SCORE_MAX_LEN, return_tensors="pt",
            ).to(llm.device)

            try:
                scores = verifier_head.score(llm, enc["input_ids"], enc["attention_mask"])
                all_scores.extend(scores.float().tolist())
            except torch.cuda.OutOfMemoryError:
                # Mini-batch was still too big — score one candidate at a time
                torch.cuda.empty_cache()
                for i in range(len(mini_batch)):
                    single_ids  = enc["input_ids"][i:i+1]
                    single_mask = enc["attention_mask"][i:i+1]
                    s = verifier_head.score(llm, single_ids, single_mask)
                    all_scores.append(s.float().item())
                    torch.cuda.empty_cache()

            del enc
            torch.cuda.empty_cache()
    finally:
        tokenizer.padding_side = _saved_side

    return all_scores


def choose_best_sample_ebm(
    item: dict,
    samples: list,
    finish_reasons: list,
    verifier_head=None,
) -> dict:
    """EBM reranking; falls back to majority vote if head is None OR if scoring OOMs."""
    if verifier_head is not None:
        try:
            scores = score_candidates_ebm(item, samples, verifier_head)
        except torch.cuda.OutOfMemoryError:
            # Last-resort fallback if even one-at-a-time scoring fails
            torch.cuda.empty_cache()
            print(f"  EBM OOM on id={item.get('id')} — falling back to majority vote")
            result = choose_best_sample(samples, finish_reasons)
            result.update({"ebm_used": False, "ebm_fallback": True, "ebm_scores": None})
            return result

        best_idx = scores.index(max(scores))
        best_ans = extract_boxed(samples[best_idx])

        if not best_ans and any(extract_boxed(s) for s in samples):
            result = choose_best_sample(samples, finish_reasons)
            result.update({"ebm_used": False, "ebm_fallback": True, "ebm_scores": scores})
            return result

        finish = finish_reasons[best_idx]
        return {
            "response":        samples[best_idx],
            "answer":          best_ans,
            "finish_reason":   finish,
            "consensus_count": 1,
            "n_samples":       len(samples),
            "uncertain":       is_uncertain(samples[best_idx], finish),
            "ebm_used":        True,
            "ebm_fallback":    False,
            "ebm_scores":      scores,
            "ebm_best_score":  scores[best_idx],
        }
    else:
        result = choose_best_sample(samples, finish_reasons)
        result.update({"ebm_used": False, "ebm_fallback": False, "ebm_scores": None})
        return result

## 5. Load Model

Loads the auto-selected model (`MODEL_LABEL` from Section 1) with HuggingFace Transformers.

- **≥ 20 GB VRAM** (A30 24 GB): BF16 + PyTorch SDPA — no quantization loss, no `flash_attn` package needed
- **< 20 GB VRAM**: NF4 4-bit via BitsAndBytes fallback

`tokenizer.padding_side = "left"` is set here — required for correct batched generation with decoder-only models.

In [6]:
# ── Load model — DSMLP-safe (mem-efficient SDPA, no CPU offload) ──────────────
# Two DSMLP-specific quirks handled here:
#  1) cuDNN's MHA kernel crashes with "mha_graph.execute" on Qwen3 shapes
#  2) EAGER attention OOMs on tight-VRAM GPUs (H100 MIG 1g.20gb): it materializes
#     the full (batch, heads, seq, seq) matrix in one shot — ~6 GB per layer at
#     batch=6, seq=4000, which combined with model+KV cache blows past 20 GB.
#
# The fix: SDPA backend with cuDNN + Flash explicitly DISABLED. SDPA falls back
# to the mem_efficient kernel which processes attention in tiles — never
# materializes the full S×S matrix, drastically lower peak memory.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

ATTN_IMPLEMENTATION = "sdpa"   # routes through mem_efficient kernel (set below)

# ── Force PyTorch SDPA to use the memory-efficient kernel ─────────────────────
# Disable cuDNN (the broken "mha_graph.execute" path) and Flash (OOMs on long
# prompts in this container). What's left: mem_efficient (tiled, low memory)
# and math (also tiled, last-resort fallback). Order matters — SDPA picks the
# first ENABLED backend that supports the given input shapes.
torch.backends.cuda.enable_cudnn_sdp(False)         # broken on DSMLP
torch.backends.cuda.enable_flash_sdp(False)         # OOMs on long prompts here
torch.backends.cuda.enable_mem_efficient_sdp(True)  # primary backend (tiled)
torch.backends.cuda.enable_math_sdp(True)           # math fallback (also safe)
print("SDPA backends: cuDNN=OFF  flash=OFF  mem_eff=ON  math=ON")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

_vram_gb  = (torch.cuda.get_device_properties(0).total_memory / 1e9
             if torch.cuda.is_available() else 0)
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU: {_gpu_name} | VRAM: {_vram_gb:.1f} GB")


def _load_bf16():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map={"": 0},          # everything on GPU 0, no CPU offload
        trust_remote_code=True,
        attn_implementation=ATTN_IMPLEMENTATION,
    )


def _load_nf4():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map={"": 0},
        trust_remote_code=True,
        attn_implementation=ATTN_IMPLEMENTATION,
    )


if _vram_gb >= 23:
    try:
        llm = _load_bf16()
        _qlabel = "BF16+SDPA(mem_eff)"
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("BF16 OOM'd — falling back to NF4 quantization")
        llm = _load_nf4()
        _qlabel = "NF4+SDPA(mem_eff)"
else:
    print(f"VRAM {_vram_gb:.1f} GB is tight for BF16 — using NF4 quantization")
    llm = _load_nf4()
    _qlabel = "NF4+SDPA(mem_eff)"

llm.eval()

# Sanity check: any layer on CPU? (would make generation ~100× slower)
_dev_map = getattr(llm, "hf_device_map", None)
if _dev_map:
    _cpu_layers = [k for k, v in _dev_map.items() if str(v) == "cpu"]
    if _cpu_layers:
        print(f"!! WARNING: {len(_cpu_layers)} modules on CPU (e.g. {_cpu_layers[:3]})")
    else:
        print(f"All {len(_dev_map)} modules on GPU — generation will run at full speed")

print(f"Model loaded: {MODEL_LABEL} ({_qlabel}) | "
      f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

SDPA backends: cuDNN=OFF  flash=OFF  mem_eff=ON  math=ON


GPU: NVIDIA RTX A5000 | VRAM: 25.4 GB


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:  21%|██        | 84/398 [00:00<00:00, 836.70it/s]

Loading weights:  42%|████▏     | 168/398 [00:00<00:00, 769.40it/s]

Loading weights:  69%|██████▉   | 276/398 [00:00<00:00, 895.86it/s]

Loading weights:  92%|█████████▏| 367/398 [00:00<00:00, 859.75it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 859.05it/s]

Model loaded: qlora_v1_merged (BF16+SDPA(mem_eff)) | VRAM: 2.65 GB allocated


In [7]:
# ── Load EBM verifier head ────────────────────────────────────────────────────
# Trained by notebooks/05_train_ebm_verifier.ipynb.
# Backbone = llm already loaded above — ZERO additional VRAM for weights (~10 KB head).
# EBM ranks whatever PHASE2_N_SAMPLES is set to in the config cell (default 3).
# We do NOT auto-bump to 8 here — that would 2.7× Phase 2 time on the MIG slice
# without proportional accuracy gain. 3 candidates is enough for EBM reranking
# to outperform majority vote in most cases.

verifier_head = None

if USE_EBM and (EBM_PATH / "verifier_head.pt").is_file():
    try:
        _vcfg = json.loads((EBM_PATH / "verifier_config.json").read_text())
        verifier_head = VerifierHead(llm.config.hidden_size).to(llm.device).to(torch.bfloat16)
        verifier_head.load_state_dict(
            torch.load(EBM_PATH / "verifier_head.pt", map_location=llm.device)
        )
        verifier_head.eval()
        print(f"Loaded EBM verifier head ({_vcfg.get('type', 'ebm_v1')}) | "
              f"Will rerank {PHASE2_N_SAMPLES} Phase 2 candidates per question")
        # Auto-bump to 8 DISABLED — Phase 2 stays at PHASE2_N_SAMPLES from config (3).
        # The bigger N would help slightly but adds ~62% to Phase 2 time on a
        # compute-limited GPU like the H100 MIG 1g.20gb. Not worth it under time pressure.
    except Exception as _e:
        print(f"EBM load failed ({_e}) — falling back to majority vote")
        verifier_head = None
else:
    print("EBM verifier head not found — using majority vote "
          "(train it with 05_train_ebm_verifier.ipynb after the full public run)")

Loaded EBM verifier head (ebm_bradley_terry_v1) | Will rerank 3 Phase 2 candidates per question


## 6. Generate Responses — Adaptive Three-Phase

Cascading retry strategy. Each phase only processes questions that are still uncertain after the previous phase, so total compute scales with question difficulty rather than dataset size.

1. **Phase 1 — fast full sweep:** every question gets one response. Short thinking budget (`PHASE1_THINKING_BUDGET=1024`), `max_new_tokens=PHASE1_MAX_TOKENS`. Chunked in groups of `CHUNK_SIZE`, checkpoint written after every chunk.

2. **Phase 2 — targeted retry:** questions flagged uncertain in Phase 1 get `PHASE2_N_SAMPLES=3` candidates with the full Phase 2 thinking budget. EBM verifier reranks them (or majority vote if EBM not loaded). Checkpoint written after every question.

3. **Phase 3 — last-resort retry** *(enabled by `USE_PHASE3 = True`)*: questions **still** flagged uncertain after Phase 2 get `PHASE3_N_SAMPLES=8` candidates with the maximum thinking budget, higher temperature for diversity, and stronger repetition penalty. EBM reranks. This is the most expensive phase but recovers some of the genuinely hard questions.

The checkpoint file (`private_*_checkpoint.jsonl`) allows the run to resume from where it left off if the pod is interrupted.

In [8]:
# ── Adaptive three-phase generation (HuggingFace Transformers path) ───────────
# Phase 1: one fast response per question, CHUNK_SIZE questions per batch.
# Phase 2: retry uncertain questions, PHASE2_BATCH_QUESTIONS questions × N samples
#          per batched generate() call. EBM ranks each question's samples.
# Phase 3 (optional, USE_PHASE3): retry STILL-uncertain questions with N=8 samples.
# Checkpoints: every chunk in Phase 1, every batched Phase 2/3 call.

response_records = load_checkpoint(CHECKPOINT_PATH)

# ── Phase 1 ────────────────────────────────────────────────────────────────────
phase1_params  = make_sampling_params(PHASE1_MAX_TOKENS, temperature=0.6)
missing_phase1 = [item for item in data_run if str(item["id"]) not in response_records]
print(f"Phase 1: {len(missing_phase1)} missing / {len(data_run)} total "
      f"({len(data_run) - len(missing_phase1)} checkpointed)")

with tqdm(total=len(missing_phase1), desc="Phase 1", unit="q") as pbar:
    for chunk_start in range(0, len(missing_phase1), CHUNK_SIZE):
        batch   = missing_phase1[chunk_start : chunk_start + CHUNK_SIZE]
        prompts = [build_chat_prompt(item, thinking_budget=PHASE1_THINKING_BUDGET)
                   for item in batch]
        outputs = generate_batch(prompts, phase1_params, chunk_size=len(batch))

        for item, out in zip(batch, outputs):
            response = out["text"]
            finish   = out["finish_reason"]
            response_records[str(item["id"])] = {
                "id": item["id"], "phase_used": 1, "response": response,
                "answer": extract_boxed(response), "finish_reason": finish,
                "uncertain": is_uncertain(response, finish),
                "n_samples": 1, "consensus_count": 1,
                "ebm_used": False,
            }
        write_checkpoint(CHECKPOINT_PATH, response_records)
        pbar.update(len(batch))

p1_uncertain = sum(1 for item in data_run
                   if response_records.get(str(item["id"]), {}).get("uncertain"))
print(f"Phase 1 done: {p1_uncertain}/{len(data_run)} uncertain")

# ── Phase 2 — cross-question batched ──────────────────────────────────────────
# Builds PHASE2_BATCH_QUESTIONS prompts × PHASE2_N_SAMPLES copies each → one
# batched generate() call. Then splits outputs per question and EBM-ranks each.
# Resume-safe: filter already includes phase_used < 2, so done Phase 2 items skip.
phase2_params    = make_sampling_params(PHASE2_MAX_TOKENS, temperature=PHASE2_TEMPERATURE,
                                         repetition_penalty=PHASE2_REPETITION_PENALTY)
phase1_uncertain = [
    item for item in data_run
    if response_records.get(str(item["id"]), {}).get("uncertain")
    and int(response_records.get(str(item["id"]), {}).get("phase_used", 0)) < 2
]
RETRY_PREFIX      = "Previous attempt was unclear. Solve this again carefully from scratch:\n\n"
MCQ_VERIFY_SUFFIX = (
    "\n\nAfter finding your answer, check each option against the problem conditions. "
    "Eliminate any letter that clearly fails. "
    "Then on the very last line write ONLY \\boxed{X}."
)

if phase1_uncertain:
    _seqs_per_call = PHASE2_BATCH_QUESTIONS * PHASE2_N_SAMPLES
    print(f"Phase 2: {len(phase1_uncertain)} uncertain × {PHASE2_N_SAMPLES} samples"
          f"  [{'EBM reranking' if verifier_head is not None else 'majority vote'}]"
          f"  (batched: {PHASE2_BATCH_QUESTIONS} questions × {PHASE2_N_SAMPLES} = "
          f"{_seqs_per_call} sequences per generate() call)")

    pbar = tqdm(total=len(phase1_uncertain), desc="Phase 2", unit="q")

    for batch_start in range(0, len(phase1_uncertain), PHASE2_BATCH_QUESTIONS):
        batch_items = phase1_uncertain[batch_start : batch_start + PHASE2_BATCH_QUESTIONS]

        # Build K prompts, replicate each N times → K×N total prompts
        all_prompts = []
        for item in batch_items:
            suffix = MCQ_VERIFY_SUFFIX if item.get("options") else ""
            prompt_text = build_chat_prompt(
                item, thinking_budget=PHASE2_THINKING_BUDGET,
                prefix=RETRY_PREFIX, suffix=suffix,
            )
            all_prompts.extend([prompt_text] * PHASE2_N_SAMPLES)

        # ONE batched generate() call for all K×N sequences
        # (chunk_size=len(all_prompts) keeps them in one call rather than splitting)
        outputs = generate_batch(all_prompts, phase2_params, chunk_size=len(all_prompts))

        # Split outputs back per question, EBM-rank each question's samples
        for q_idx, item in enumerate(batch_items):
            s_idx = q_idx * PHASE2_N_SAMPLES
            e_idx = s_idx + PHASE2_N_SAMPLES
            samples        = [outputs[i]["text"]          for i in range(s_idx, e_idx)]
            finish_reasons = [outputs[i]["finish_reason"] for i in range(s_idx, e_idx)]

            # Archive Phase 1 record for analysis (and potential EBM retraining)
            _p1_rec = response_records.get(str(item["id"]), {})

            chosen = choose_best_sample_ebm(
                item=item,
                samples=samples,
                finish_reasons=finish_reasons,
                verifier_head=verifier_head,
            )
            chosen.update({
                "id":              item["id"],
                "phase_used":      2,
                "phase1_response": _p1_rec.get("response", ""),
                "phase1_answer":   _p1_rec.get("answer",   ""),
            })
            response_records[str(item["id"])] = chosen

        # Checkpoint after every batched call (worst case lose PHASE2_BATCH_QUESTIONS items)
        write_checkpoint(CHECKPOINT_PATH, response_records)
        pbar.update(len(batch_items))

    pbar.close()

p2_uncertain = sum(1 for item in data_run
                   if response_records.get(str(item["id"]), {}).get("uncertain"))
print(f"Phase 2 done: {p2_uncertain}/{len(data_run)} still uncertain")

# ── Phase 3 — DISABLED by default (USE_PHASE3=False) ─────────────────────────
# When enabled, retries questions STILL flagged uncertain after Phase 2 with
# N=PHASE3_N_SAMPLES, higher temperature, larger token budget. EBM reranks.
if USE_PHASE3:
    phase3_params    = make_sampling_params(
        PHASE3_MAX_TOKENS,
        temperature=PHASE3_TEMPERATURE,
        repetition_penalty=PHASE3_REPETITION_PENALTY,
    )
    phase2_uncertain = [
        item for item in data_run
        if response_records.get(str(item["id"]), {}).get("uncertain")
        and int(response_records.get(str(item["id"]), {}).get("phase_used", 0)) < 3
    ]
    PHASE3_PREFIX = (
        "Two previous attempts were unclear. This is a difficult problem — "
        "take your time, consider multiple solution approaches, work through "
        "the math very carefully step by step, and double-check your answer "
        "before writing the final boxed result.\n\n"
    )

    if phase2_uncertain:
        print(f"Phase 3: {len(phase2_uncertain)} STILL uncertain × {PHASE3_N_SAMPLES} samples"
              f"  [{'EBM reranking' if verifier_head is not None else 'majority vote'}]")
        for item in tqdm(phase2_uncertain, desc="Phase 3", unit="q"):
            suffix      = MCQ_VERIFY_SUFFIX if item.get("options") else ""
            prompt_text = build_chat_prompt(
                item, thinking_budget=PHASE3_THINKING_BUDGET,
                prefix=PHASE3_PREFIX, suffix=suffix,
            )
            outputs        = generate_batch([prompt_text] * PHASE3_N_SAMPLES, phase3_params)
            samples        = [o["text"]          for o in outputs]
            finish_reasons = [o["finish_reason"] for o in outputs]

            _p2_rec = response_records.get(str(item["id"]), {})

            chosen = choose_best_sample_ebm(
                item=item,
                samples=samples,
                finish_reasons=finish_reasons,
                verifier_head=verifier_head,
            )
            chosen.update({
                "id":              item["id"],
                "phase_used":      3,
                "phase1_response": _p2_rec.get("phase1_response", ""),
                "phase1_answer":   _p2_rec.get("phase1_answer",   ""),
                "phase2_response": _p2_rec.get("response", ""),
                "phase2_answer":   _p2_rec.get("answer",   ""),
            })
            response_records[str(item["id"])] = chosen
            write_checkpoint(CHECKPOINT_PATH, response_records)

        p3_uncertain = sum(1 for item in data_run
                           if response_records.get(str(item["id"]), {}).get("uncertain"))
        print(f"Phase 3 done: {p3_uncertain}/{len(data_run)} still uncertain")
    else:
        print("Phase 3: no questions still uncertain after Phase 2 — skipped")

# ── Final assembly ─────────────────────────────────────────────────────────────
missing_ids = [item["id"] for item in data_run if str(item["id"]) not in response_records]
if missing_ids:
    raise RuntimeError(f"Missing responses for {len(missing_ids)} id(s): {missing_ids[:10]}")

responses     = [response_records[str(item["id"])]["response"] for item in data_run]
phase_counts  = Counter(response_records[str(item["id"])].get("phase_used") for item in data_run)
uncertain     = sum(bool(response_records[str(item["id"])].get("uncertain")) for item in data_run)
finish_counts = Counter(str(response_records[str(item["id"])].get("finish_reason")) for item in data_run)
ebm_count     = sum(bool(response_records[str(item["id"])].get("ebm_used")) for item in data_run)

print(f"\nDone: {len(responses)} responses | "
      f"phase1={phase_counts.get(1,0)} phase2={phase_counts.get(2,0)} "
      f"phase3={phase_counts.get(3,0)} "
      f"uncertain={uncertain} ebm_used={ebm_count}")
print(f"Finish reasons: {dict(finish_counts)}")

Loaded 943 checkpoint records from private_qlora_v1_merged_checkpoint.jsonl
Phase 1: 0 missing / 943 total (943 checkpointed)


Phase 1: 0q [00:00, ?q/s]

Phase 1: 0q [00:00, ?q/s]

Phase 1 done: 520/943 uncertain
Phase 2: 232 uncertain × 3 samples  [EBM reranking]  (batched: 3 questions × 3 = 9 sequences per generate() call)


Phase 2:   0%|          | 0/232 [00:00<?, ?q/s]

...

Phase 2:   9%|▉         | 21/232 [1:45:23<18:16:12, 311.72s/q]

Phase 2:  10%|█         | 24/232 [1:59:49<17:35:02, 304.34s/q]

Phase 2:  12%|█▏        | 27/232 [2:15:06<17:21:15, 304.76s/q]

Phase 2:  13%|█▎        | 30/232 [2:30:23<17:06:55, 305.03s/q]

Phase 2:  14%|█▍        | 33/232 [2:44:33<16:29:40, 298.39s/q]

## 7. Save Results & Export CSV

Writes two files:

| File | Format | Purpose |
|------|--------|---------|
| `artifacts/logs/runs/private_*_results.jsonl` | JSONL | Full metadata per question (phase, finish reason, consensus count, model label) |
| `artifacts/submissions/submission_*_YYYY-MM-DD.csv` | CSV (`id`, `response`) | Competition submission — `QUOTE_ALL` to handle commas/newlines in LaTeX responses |

In [ ]:
import csv
from datetime import date

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item, response in zip(data_run, responses):
        meta = response_records.get(str(item["id"]), {})
        f.write(json.dumps({
            "id":              item["id"],
            "is_mcq":          bool(item.get("options")),
            "response":        response,
            "phase_used":      meta.get("phase_used"),
            "uncertain":       meta.get("uncertain"),
            "finish_reason":   meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples":       meta.get("n_samples"),
            "model":           MODEL_LABEL,
        }, ensure_ascii=False) + "\n")

SUBMISSION_PATH = (
    REPO_ROOT / "artifacts" / "submissions"
    / f"submission_{MODEL_LABEL}_{date.today().isoformat()}.csv"
)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)

rows = sorted(
    [{"id": item["id"], "response": response_records[str(item["id"])]["response"]}
     for item in data_run],
    key=lambda r: r["id"],
)

with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(rows)

print(f"JSONL : {OUTPUT_PATH}")
print(f"CSV   : {SUBMISSION_PATH}  ({len(rows)} rows)")
